In [3]:
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

# ========= 按你的环境改这几行 =========
# OpenRouter 控制台 Keys，一般形如 sk-or-v1-...（不要用字面量 "xxx"，否则会 401 Missing Authentication）
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "mistral1.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "mistralai/mistral-large"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "mistral-large"  # 写入 CSV 的 model 列
PROMPT_TYPE = "baseline"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- The question should be clear and relevant
- The answer should be concise and accurate
- Do not add information that is not in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")


--- chunk_id=1 (1/104) ---
Question: What is the time frame for returning an undamaged product to Apple under the standard return policy?

Answer: 14 days from the date you receive the product.

--- chunk_id=2 (2/104) ---
Question: What happens if a customer returns an item purchased with an Apple Account balance that is at or near the maximum limit?
Answer: Apple will issue the refund amount on an Apple Gift Card by email.

--- chunk_id=3 (3/104) ---
Question: What condition must an Apple Watch from the Edition collection meet to be eligible for return or exchange?
Answer: It must be in its original, undamaged, and unmarked condition after passing inspection at Apple's offsite facility.

--- chunk_id=4 (4/104) ---
Question: What happens if Apple reduces the price of a product within 14 days of purchase?
Answer: You can request a refund or credit for the difference between the price you paid and the current selling price, provided you contact Apple within 14 calendar days of the price 

In [5]:
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "mistral2.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "mistralai/mistral-large"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "mistral-large"  # 写入 CSV 的 model 列
PROMPT_TYPE = "concise"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- The question should focus on an important point
- The answer should be brief and factual
- Do not add information that is not in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")


--- chunk_id=1 (1/104) ---
Question: What is the time limit for returning an undamaged product to Apple under the standard return policy?

Answer: 14 days from the date you receive the product.

--- chunk_id=2 (2/104) ---
Question: What happens if a customer returns an item purchased with an Apple Account balance?
Answer: The refund for that portion is issued back to the Apple Account balance, or as an Apple Gift Card by email if the balance is at or near the maximum limit.

--- chunk_id=3 (3/104) ---
Question: What condition must an Apple Watch from the Edition collection meet to be returned or exchanged?
Answer: It must be in its original, undamaged, and unmarked condition after inspection.

--- chunk_id=4 (4/104) ---
Question: What happens if an iPhone is damaged due to unauthorized software modifications like jailbreaking?
Answer: Its repair will not be covered under the warranty.

--- chunk_id=5 (5/104) ---
Question: What is the maximum number of units eligible for price protectio

In [7]:
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "mistral3.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "mistralai/mistral-large"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "mistral-large"  # 写入 CSV 的 model 列
PROMPT_TYPE = "key_information"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- Focus on the most important idea in the text
- Avoid minor or trivial details
- The answer should be clear, accurate, and informative
- Do not add information that is not stated in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")

--- chunk_id=1 (1/104) ---
Question: What is Apple's standard return window for undamaged products purchased from the Apple Store?

Answer: Apple's standard return policy allows customers to return undamaged products with their included accessories, packaging, and original receipt within **14 days** of receiving the product.

--- chunk_id=2 (2/104) ---
**Question:** What are the key conditions for returning a product to Apple?

**Answer:** Products can only be returned in the country or region where they were originally purchased. Certain items—such as electronic software downloads, subscriptions, Apple Store Gift Cards, and Apple Developer Connection products—are not eligible for return. Additionally, opened software with a visible license seal cannot be returned unless it is Apple-branded software that the customer does not agree to the licensing terms for. For returns of ten or more of the same product, the return must be made to the original Apple Store of purchase.

--- chunk_id=3